# MedAssist AI — Medical Q&A Chatbot using MedQuAD
**Run in Google Colab, top to bottom.** This starter builds a semantic retrieval chatbot from the MedQuAD XML files, exports the knowledge base, and creates a Streamlit app.

**Safety:** educational prototype only; not a diagnosis or treatment tool. For emergencies, contact local emergency services. Do not upload private patient data.

Dataset: https://github.com/abachaa/MedQuAD


## 1. Install dependencies

In [ ]:
!pip -q install sentence-transformers faiss-cpu pandas streamlit


## 2. Download MedQuAD

In [ ]:
!git clone --depth 1 https://github.com/abachaa/MedQuAD.git


## 3. Parse the XML files into question-answer records

In [ ]:
import os, glob, xml.etree.ElementTree as ET
import pandas as pd

ROOT = "/content/MedQuAD"
records = []

def find_text(el, names):
    for node in el.iter():
        tag = node.tag.split("}")[-1].lower()
        if tag in names and node.text and node.text.strip():
            return " ".join(node.text.split())
    return ""

for path in glob.glob(os.path.join(ROOT, "**", "*.xml"), recursive=True):
    try:
        root = ET.parse(path).getroot()
    except Exception:
        continue
    for qa in root.iter():
        if qa.tag.split("}")[-1].lower() != "qapair":
            continue
        question = find_text(qa, {"question"})
        answer = find_text(qa, {"answer"})
        if question and answer:
            records.append({
                "question": question,
                "answer": answer,
                "source_file": os.path.relpath(path, ROOT)
            })

df = pd.DataFrame(records).drop_duplicates(subset=["question", "answer"])
df = df[df["answer"].str.len() > 15].reset_index(drop=True)
print("Parsed Q&A pairs:", len(df))
display(df.head())
if len(df) == 0:
    raise ValueError("No Q&A pairs were parsed. Check the cloned repository/XML structure.")


## 4. Build a semantic vector index

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(MODEL_NAME)

# Embed the question and answer together so searches can match either concept.
documents = (df["question"] + "\n" + df["answer"].str.slice(0, 1800)).tolist()
embeddings = embedder.encode(
    documents, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True, batch_size=64
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])  # cosine similarity with normalized vectors
index.add(embeddings)

OUT = "/content/medassist_output"
os.makedirs(OUT, exist_ok=True)
df.to_csv(os.path.join(OUT, "medquad_qa.csv"), index=False)
faiss.write_index(index, os.path.join(OUT, "medquad.faiss"))
with open(os.path.join(OUT, "model_name.txt"), "w") as f:
    f.write(MODEL_NAME)

print(f"Saved {index.ntotal} entries to {OUT}")


## 5. Test retrieval

In [ ]:
def retrieve(question, k=3):
    qv = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            row = df.iloc[int(idx)]
            results.append({"score": float(score), "question": row["question"],
                            "answer": row["answer"], "source": row["source_file"]})
    return results

for item in retrieve("What are common symptoms of diabetes?", 3):
    print("\\nSimilarity:", round(item["score"], 3))
    print("Matched question:", item["question"])
    print("Answer:", item["answer"][:600])
    print("Source:", item["source"])


## 6. Create the Streamlit app

In [ ]:
%%writefile /content/medassist_output/app.py
import os, re
import pandas as pd
import numpy as np
import faiss
import streamlit as st
from sentence_transformers import SentenceTransformer

st.set_page_config(page_title="MedAssist AI", page_icon="🩺", layout="wide")
st.title("🩺 MedAssist AI")
st.caption("Medical Q&A retrieval prototype powered by the MedQuAD dataset")

DATA_DIR = os.path.dirname(os.path.abspath(__file__))
CSV_PATH = os.path.join(DATA_DIR, "medquad_qa.csv")
INDEX_PATH = os.path.join(DATA_DIR, "medquad.faiss")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

@st.cache_resource
def load_assets():
    data = pd.read_csv(CSV_PATH).fillna("")
    vec_index = faiss.read_index(INDEX_PATH)
    model = SentenceTransformer(MODEL_NAME)
    return data, vec_index, model

df, index, model = load_assets()

def recognize_entities(text):
    terms = {
        "Symptoms": ["fever", "cough", "headache", "pain", "fatigue", "nausea",
                     "vomiting", "rash", "dizziness", "shortness of breath"],
        "Diseases/conditions": ["diabetes", "asthma", "cancer", "hypertension",
                                "migraine", "arthritis", "depression", "anemia"],
        "Treatments/medicines": ["treatment", "therapy", "medication", "medicine",
                                 "antibiotic", "chemotherapy", "insulin", "vaccine"]
    }
    low = text.lower()
    return {category: [term for term in words if re.search(r"\\b" + re.escape(term) + r"\\b", low)]
            for category, words in terms.items()}

def retrieve(question, k=3):
    qv = model.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    out = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            row = df.iloc[int(idx)]
            out.append((float(score), row))
    return out

st.info("For education only. This tool cannot diagnose conditions or replace a qualified clinician.")
question = st.text_input("Ask a medical question", placeholder="Example: What are common symptoms of diabetes?")
k = st.slider("Number of retrieved answers", min_value=1, max_value=5, value=3)

if st.button("Find answer", type="primary") and question.strip():
    with st.spinner("Searching MedQuAD..."):
        results = retrieve(question.strip(), k)
    entities = recognize_entities(question)
    st.subheader("Medical terms detected")
    any_entity = False
    for category, values in entities.items():
        if values:
            any_entity = True
            st.write(f"**{category}:** " + ", ".join(values))
    if not any_entity:
        st.write("No terms found by the basic keyword recognizer.")
    if not results or results[0][0] < 0.20:
        st.warning("I could not find a sufficiently relevant answer in the current dataset. Try rephrasing or consult a healthcare professional.")
    else:
        st.subheader("Best matching answer")
        score, row = results[0]
        st.write(row["answer"])
        st.caption(f"Retrieval similarity: {score:.3f} — this is a ranking score, not medical confidence.")
        st.caption("Matched dataset question: " + str(row["question"]))
        st.caption("Dataset file: " + str(row.get("source_file", "MedQuAD")))
        with st.expander("Other relevant matches"):
            for rank, (s, r) in enumerate(results[1:], start=2):
                st.markdown(f"**Match {rank} · similarity {s:.3f}**")
                st.write(r["question"])
                st.write(str(r["answer"])[:1200])
                st.caption("Source file: " + str(r.get("source_file", "MedQuAD")))

st.divider()
st.subheader("Update knowledge base")
st.write("For a demo, upload a CSV with columns `question` and `answer`. This app displays how new knowledge can be added; rebuilding embeddings/index is required before new records become searchable.")
uploaded = st.file_uploader("Upload additional Q&A CSV", type=["csv"])
if uploaded is not None:
    try:
        extra = pd.read_csv(uploaded).fillna("")
        if not {"question", "answer"}.issubset(extra.columns):
            st.error("CSV must contain question and answer columns.")
        else:
            st.success(f"Validated {len(extra)} new Q&A rows. Save these rows and rebuild the vector index to activate them.")
            st.dataframe(extra[["question", "answer"]].head(10), use_container_width=True)
    except Exception as e:
        st.error(f"Could not read CSV: {e}")

st.caption("Prototype only. Do not use it to make clinical decisions. If symptoms are severe or urgent, seek emergency care.")


## 7. Package the app files into a ZIP

In [ ]:
import shutil, os
zip_path = "/content/MedAssist_AI_Submission.zip"
shutil.make_archive("/content/MedAssist_AI_Submission", "zip", "/content/medassist_output")
print("Created:", zip_path)
from google.colab import files
files.download(zip_path)


## 8. Run Streamlit (optional live demo)
Colab is a notebook environment, not a permanent hosting service. The ZIP is the reliable submission artifact. To demonstrate the UI, run the following cell; if the tunnel does not start in your session, submit `app.py` and the ZIP instead.

```python
!npm -q install -g localtunnel
%cd /content/medassist_output
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /tmp/streamlit.log 2>&1 &
!lt --port 8501
```

Keep the cell running and open the public URL printed by LocalTunnel. Do not put private patient information into the demo.
